In [1]:
# =================================================================================================
# NHANES 2015-2016 DIETARY MERGE - FIDELITY PRESERVING
#
# Source files:
#
#   DR1TOT_I_corrected.csv
#   DR2TOT_I_corrected.csv
#
#
# Purpose:
#
#   Merge NHANES 2015-2016 Dietary Day 1 and Day 2 Total Nutrient
#   Intake files into one participant-level corrected Dietary CSV.
#
#
# FIDELITY PRINCIPLES
# -------------------------------------------------------------------------------------------------
#
#   1. Read the already-corrected CSV files AS TEXT.
#
#   2. Preserve Dietary decimal representation AS IS.
#
#   3. Do NOT round.
#
#   4. Do NOT use float_format=.
#
#   5. Do NOT integerize nutrient measurements.
#
#   6. Preserve legitimate numeric zeros.
#
#   7. Validate shared Day 1 / Day 2 variables before removing duplicate copies.
#
#   8. Retain ONE canonical copy of:
#
#          WTDRD1
#          WTDR2D
#          DRABF
#          DRDINT
#
#   9. Prevent engineered duplicate fields such as:
#
#          WTDRD1_DAY_2_TOTAL_INTAKE
#          WTDR2D_DAY_2_TOTAL_INTAKE
#          DRABF_DAY_2_TOTAL_INTAKE
#          DRDINT_DAY_2_TOTAL_INTAKE
#
#  10. Every participant-level merge must be one-to-one.
#
#  11. Validate source-to-merged exact text fidelity.
#
#  12. Validate exact pre-save/post-save text fidelity.
#
#
# CDC 2015-2016 BENCHMARKS
# -------------------------------------------------------------------------------------------------
#
#   DR1TOT_I rows                 = 9,544
#   DR2TOT_I rows                 = 9,544
#
#
#   WTDRD1:
#
#       positive Day 1 weight     = 8,506
#       zero                      = 1,038
#       missing                   =     0
#
#
#   WTDR2D:
#
#       positive two-day weight   = 7,027
#       zero                      = 1,479
#       missing                   = 1,038
#
#
#   DRDINT:
#
#       1 = Day 1 only            = 1,479
#       2 = Day 1 + Day 2         = 7,027
#       missing                   = 1,038
#
#
#   DRABF:
#
#       1 = Yes                   =   180
#       2 = No                    = 8,326
#       missing                   = 1,038
#
# =================================================================================================


# =================================================================================================
# 1. IMPORTS
# =================================================================================================

import pandas as pd
import numpy as np
from pathlib import Path


# =================================================================================================
# 2. PATHS
# =================================================================================================

BASE_DIR = Path().resolve().parent


DATA_DIR = (
    BASE_DIR
    / "data"
    / "2015_2016"
    / "Dietary"
)


DAY1_FILE = (
    DATA_DIR
    / "DR1TOT_I_corrected.csv"
)


DAY2_FILE = (
    DATA_DIR
    / "DR2TOT_I_corrected.csv"
)


OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "processed_v3"
    / "2015_2016"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


OUTPUT_PATH = (
    OUTPUT_DIR
    / "neighbourhood_nhanes_dietary_2015_2016_raw_corrected.csv"
)


# =================================================================================================
# 3. VERIFY FILES EXIST
# =================================================================================================

if not DAY1_FILE.exists():

    raise FileNotFoundError(
        f"Cannot find Day 1 file:\n{DAY1_FILE}"
    )


if not DAY2_FILE.exists():

    raise FileNotFoundError(
        f"Cannot find Day 2 file:\n{DAY2_FILE}"
    )


# =================================================================================================
# 4. READ CORRECTED FILES AS TEXT
# =================================================================================================

#
# CRITICAL:
#
# dtype=str prevents pandas from re-parsing and reformatting
# already-corrected Dietary decimal values.
#
#
# Example:
#
#     0.028999999999
#
# stays:
#
#     0.028999999999
#
#
# This notebook is performing a MERGE.
#
# It is NOT recalculating Dietary nutrient values.
#

day1 = pd.read_csv(
    DAY1_FILE,
    dtype=str,
    keep_default_na=False
)


day2 = pd.read_csv(
    DAY2_FILE,
    dtype=str,
    keep_default_na=False
)


print("=" * 110)

print(
    "NHANES 2015-2016 DIETARY DAY 1 + DAY 2 MERGE"
)

print("=" * 110)


print(
    "DR1TOT_I shape:",
    day1.shape
)


print(
    "DR2TOT_I shape:",
    day2.shape
)


# =================================================================================================
# 5. STANDARDIZE COLUMN NAMES ONLY
# =================================================================================================

#
# Values themselves are NOT modified.
#

day1.columns = (
    day1.columns
    .str.strip()
    .str.upper()
)


day2.columns = (
    day2.columns
    .str.strip()
    .str.upper()
)


# =================================================================================================
# 6. CYCLE-SPECIFIC CDC BENCHMARKS
# =================================================================================================

EXPECTED_ROWS = 9544


# -------------------------------------------------------------------------------------------------
# WTDRD1
# -------------------------------------------------------------------------------------------------

EXPECTED_WTDRD1_POSITIVE = 8506

EXPECTED_WTDRD1_ZERO = 1038

EXPECTED_WTDRD1_MISSING = 0


# -------------------------------------------------------------------------------------------------
# WTDR2D
# -------------------------------------------------------------------------------------------------

EXPECTED_WTDR2D_POSITIVE = 7027

EXPECTED_WTDR2D_ZERO = 1479

EXPECTED_WTDR2D_MISSING = 1038


# -------------------------------------------------------------------------------------------------
# DRDINT
# -------------------------------------------------------------------------------------------------

EXPECTED_DRDINT = {

    "1": 1479,

    "2": 7027,

    "": 1038

}


# -------------------------------------------------------------------------------------------------
# DRABF
# -------------------------------------------------------------------------------------------------

EXPECTED_DRABF = {

    "1": 180,

    "2": 8326,

    "": 1038

}


# =================================================================================================
# 7. VERIFY SEQN EXISTS
# =================================================================================================

for name, dataframe in [

    ("DR1TOT_I", day1),

    ("DR2TOT_I", day2)

]:

    if "SEQN" not in dataframe.columns:

        raise ValueError(
            f"{name} does not contain SEQN."
        )


print(
    "PASS: SEQN exists in both files."
)


# =================================================================================================
# 8. VERIFY OFFICIAL ROW COUNTS
# =================================================================================================

if len(day1) != EXPECTED_ROWS:

    raise ValueError(
        "Unexpected DR1TOT_I row count.\n"
        f"Expected: {EXPECTED_ROWS:,}\n"
        f"Actual:   {len(day1):,}"
    )


if len(day2) != EXPECTED_ROWS:

    raise ValueError(
        "Unexpected DR2TOT_I row count.\n"
        f"Expected: {EXPECTED_ROWS:,}\n"
        f"Actual:   {len(day2):,}"
    )


print(
    "PASS: Both DR1TOT_I and DR2TOT_I "
    "contain 9,544 participant rows."
)


# =================================================================================================
# 9. SEQN VALIDATION FUNCTION
# =================================================================================================

def validate_seqn(
    dataframe,
    name
):

    print(
        f"\n--- SEQN CHECK: {name} ---"
    )


    blank_count = int(

        dataframe["SEQN"]
        .eq("")
        .sum()

    )


    duplicate_count = int(

        dataframe["SEQN"]
        .duplicated()
        .sum()

    )


    dot_zero_count = int(

        (
            dataframe["SEQN"].ne("")
            &
            dataframe["SEQN"]
            .str.endswith(".0")
        )

        .sum()

    )


    invalid_format_count = int(

        (
            dataframe["SEQN"].ne("")
            &
            ~dataframe["SEQN"]
            .str.fullmatch(r"\d+")
        )

        .sum()

    )


    print(
        "Rows:",
        f"{len(dataframe):,}"
    )


    print(
        "Unique SEQN:",
        f"{dataframe['SEQN'].nunique():,}"
    )


    print(
        "Blank SEQN:",
        blank_count
    )


    print(
        "Duplicate SEQN:",
        duplicate_count
    )


    print(
        "SEQN ending '.0':",
        dot_zero_count
    )


    print(
        "Invalid SEQN:",
        invalid_format_count
    )


    if blank_count != 0:

        raise ValueError(
            f"{name}: blank SEQN found."
        )


    if duplicate_count != 0:

        raise ValueError(
            f"{name}: duplicate SEQN found."
        )


    if dot_zero_count != 0:

        raise ValueError(
            f"{name}: SEQN contains '.0'."
        )


    if invalid_format_count != 0:

        raise ValueError(
            f"{name}: invalid SEQN formatting."
        )


    print(
        "PASS"
    )


# =================================================================================================
# 10. VALIDATE SEQN IN BOTH FILES
# =================================================================================================

validate_seqn(
    day1,
    "DR1TOT_I"
)


validate_seqn(
    day2,
    "DR2TOT_I"
)


print(
    "\nPASS: SEQN integrity validated."
)


# =================================================================================================
# 11. VERIFY SAME PARTICIPANT UNIVERSE
# =================================================================================================

day1_seqn = set(
    day1["SEQN"]
)


day2_seqn = set(
    day2["SEQN"]
)


only_day1 = (
    day1_seqn
    -
    day2_seqn
)


only_day2 = (
    day2_seqn
    -
    day1_seqn
)


print(
    "\n--- PARTICIPANT SET CHECK ---"
)


print(
    "Only in DR1TOT_I:",
    len(only_day1)
)


print(
    "Only in DR2TOT_I:",
    len(only_day2)
)


if only_day1 or only_day2:

    print(
        "\nExample Day 1-only SEQN:"
    )


    print(
        sorted(
            only_day1
        )[:20]
    )


    print(
        "\nExample Day 2-only SEQN:"
    )


    print(
        sorted(
            only_day2
        )[:20]
    )


    raise ValueError(
        "DR1TOT_I and DR2TOT_I do not contain "
        "the same participant universe."
    )


print(
    "PASS: Both files contain the same "
    "9,544 participants."
)


# =================================================================================================
# 12. DETECT SAME-NAME VARIABLES
# =================================================================================================

shared_columns_detected = [

    col

    for col in day1.columns

    if (
        col in day2.columns
        and
        col != "SEQN"
    )

]


print(
    "\n--- SHARED VARIABLES DETECTED ---"
)


print(
    shared_columns_detected
)


# =================================================================================================
# 13. EXPECTED OFFICIAL SHARED VARIABLES
# =================================================================================================

#
# CDC explicitly identifies these as same-name variables
# in Day 1 and Day 2 Dietary files.
#

SHARED_COLUMNS = [

    "WTDRD1",

    "WTDR2D",

    "DRABF",

    "DRDINT"

]


if set(
    shared_columns_detected
) != set(
    SHARED_COLUMNS
):

    raise ValueError(
        "Unexpected shared-column pattern.\n\n"
        f"Expected:\n{SHARED_COLUMNS}\n\n"
        f"Detected:\n{shared_columns_detected}"
    )


print(
    "PASS: Exactly four expected shared variables found."
)


# =================================================================================================
# 14. BUILD DAY 1 / DAY 2 SHARED-VARIABLE COMPARISON
# =================================================================================================

shared_comparison = (

    day1[
        ["SEQN"] + SHARED_COLUMNS
    ]

    .merge(

        day2[
            ["SEQN"] + SHARED_COLUMNS
        ],

        on="SEQN",

        how="outer",

        suffixes=(
            "_DAY1",
            "_DAY2"
        ),

        validate="one_to_one"

    )

)


# =================================================================================================
# 15. VALIDATE SHARED VARIABLES
# =================================================================================================

#
# IMPORTANT:
#
# Files remain TEXT.
#
# Validation uses temporary numeric copies because:
#
#     "0"
#
# and
#
#     "0.0"
#
# are numerically identical even though their raw text differs.
#
# The actual source strings are never rewritten.
#

print(
    "\n--- SHARED VARIABLE FIDELITY CHECK ---"
)


for col in SHARED_COLUMNS:

    day1_raw = (
        shared_comparison[
            f"{col}_DAY1"
        ]
    )


    day2_raw = (
        shared_comparison[
            f"{col}_DAY2"
        ]
    )


    day1_missing = (
        day1_raw.eq("")
    )


    day2_missing = (
        day2_raw.eq("")
    )


    missing_mismatch_count = int(

        (
            day1_missing
            !=
            day2_missing
        )

        .sum()

    )


    both_populated = (
        ~day1_missing
        &
        ~day2_missing
    )


    day1_numeric = pd.to_numeric(

        day1_raw.loc[
            both_populated
        ],

        errors="raise"

    )


    day2_numeric = pd.to_numeric(

        day2_raw.loc[
            both_populated
        ],

        errors="raise"

    )


    numeric_equal = np.isclose(

        day1_numeric.to_numpy(
            dtype=float
        ),

        day2_numeric.to_numpy(
            dtype=float
        ),

        rtol=0,

        atol=0

    )


    numeric_mismatch_count = int(

        (
            ~numeric_equal
        )

        .sum()

    )


    day1_text = (

        day1_raw.loc[
            both_populated
        ]

        .reset_index(
            drop=True
        )

    )


    day2_text = (

        day2_raw.loc[
            both_populated
        ]

        .reset_index(
            drop=True
        )

    )


    text_difference_count = int(

        (
            day1_text
            !=
            day2_text
        )

        .sum()

    )


    print(
        f"\n{col}"
    )


    print(
        "  Both populated:",
        f"{int(both_populated.sum()):,}"
    )


    print(
        "  Missing-position mismatches:",
        f"{missing_mismatch_count:,}"
    )


    print(
        "  True numeric mismatches:",
        f"{numeric_mismatch_count:,}"
    )


    print(
        "  Text-format differences:",
        f"{text_difference_count:,}"
    )


    if missing_mismatch_count != 0:

        raise ValueError(
            f"{col}: missing positions differ "
            "between DR1TOT_I and DR2TOT_I."
        )


    if numeric_mismatch_count != 0:

        raise ValueError(
            f"{col}: true numeric values differ "
            "between DR1TOT_I and DR2TOT_I."
        )


print(
    "\nPASS: WTDRD1, WTDR2D, DRABF and DRDINT "
    "agree between Day 1 and Day 2."
)


# =================================================================================================
# 16. CDC WTDRD1 CHECK
# =================================================================================================

#
# IMPORTANT:
#
# Zero is legitimate here.
#
# Therefore validate:
#
#     positive
#     zero
#     missing
#
# separately.
#

wtdrd1_numeric = pd.to_numeric(

    day1["WTDRD1"]
    .where(
        day1["WTDRD1"] != "",
        np.nan
    ),

    errors="raise"

)


wtdrd1_positive = int(

    (
        wtdrd1_numeric > 0
    )

    .sum()

)


wtdrd1_zero = int(

    wtdrd1_numeric

    .eq(0)

    .sum()

)


wtdrd1_missing = int(

    wtdrd1_numeric

    .isna()

    .sum()

)


print(
    "\n--- CDC WTDRD1 CHECK ---"
)


print(
    "Positive weights:",
    f"{wtdrd1_positive:,}"
)


print(
    "Zero weights:",
    f"{wtdrd1_zero:,}"
)


print(
    "Missing:",
    f"{wtdrd1_missing:,}"
)


if wtdrd1_positive != EXPECTED_WTDRD1_POSITIVE:

    raise ValueError(
        "WTDRD1 positive-weight count differs from CDC.\n"
        f"Expected: {EXPECTED_WTDRD1_POSITIVE:,}\n"
        f"Actual:   {wtdrd1_positive:,}"
    )


if wtdrd1_zero != EXPECTED_WTDRD1_ZERO:

    raise ValueError(
        "WTDRD1 zero count differs from CDC.\n"
        f"Expected: {EXPECTED_WTDRD1_ZERO:,}\n"
        f"Actual:   {wtdrd1_zero:,}"
    )


if wtdrd1_missing != EXPECTED_WTDRD1_MISSING:

    raise ValueError(
        "WTDRD1 missing count differs from CDC.\n"
        f"Expected: {EXPECTED_WTDRD1_MISSING:,}\n"
        f"Actual:   {wtdrd1_missing:,}"
    )


print(
    "PASS: WTDRD1 matches CDC exactly."
)


# =================================================================================================
# 17. CDC WTDR2D CHECK
# =================================================================================================

wtdr2d_numeric = pd.to_numeric(

    day1["WTDR2D"]
    .where(
        day1["WTDR2D"] != "",
        np.nan
    ),

    errors="raise"

)


wtdr2d_positive = int(

    (
        wtdr2d_numeric > 0
    )

    .sum()

)


wtdr2d_zero = int(

    wtdr2d_numeric

    .eq(0)

    .sum()

)


wtdr2d_missing = int(

    wtdr2d_numeric

    .isna()

    .sum()

)


print(
    "\n--- CDC WTDR2D CHECK ---"
)


print(
    "Positive weights:",
    f"{wtdr2d_positive:,}"
)


print(
    "Zero weights:",
    f"{wtdr2d_zero:,}"
)


print(
    "Missing:",
    f"{wtdr2d_missing:,}"
)


if wtdr2d_positive != EXPECTED_WTDR2D_POSITIVE:

    raise ValueError(
        "WTDR2D positive-weight count differs from CDC.\n"
        f"Expected: {EXPECTED_WTDR2D_POSITIVE:,}\n"
        f"Actual:   {wtdr2d_positive:,}"
    )


if wtdr2d_zero != EXPECTED_WTDR2D_ZERO:

    raise ValueError(
        "WTDR2D zero count differs from CDC.\n"
        f"Expected: {EXPECTED_WTDR2D_ZERO:,}\n"
        f"Actual:   {wtdr2d_zero:,}"
    )


if wtdr2d_missing != EXPECTED_WTDR2D_MISSING:

    raise ValueError(
        "WTDR2D missing count differs from CDC.\n"
        f"Expected: {EXPECTED_WTDR2D_MISSING:,}\n"
        f"Actual:   {wtdr2d_missing:,}"
    )


print(
    "PASS: WTDR2D matches CDC exactly."
)


# =================================================================================================
# 18. CDC DRDINT CHECK
# =================================================================================================

drdint_counts = (

    day1["DRDINT"]

    .value_counts(
        dropna=False
    )

    .to_dict()

)


print(
    "\n--- CDC DRDINT CHECK ---"
)


print(
    drdint_counts
)


if drdint_counts != EXPECTED_DRDINT:

    raise ValueError(
        "DRDINT distribution differs from CDC.\n\n"
        f"Expected:\n{EXPECTED_DRDINT}\n\n"
        f"Actual:\n{drdint_counts}"
    )


print(
    "PASS: DRDINT matches CDC."
)


# =================================================================================================
# 19. CDC DRABF CHECK
# =================================================================================================

drabf_counts = (

    day1["DRABF"]

    .value_counts(
        dropna=False
    )

    .to_dict()

)


print(
    "\n--- CDC DRABF CHECK ---"
)


print(
    drabf_counts
)


if drabf_counts != EXPECTED_DRABF:

    raise ValueError(
        "DRABF distribution differs from CDC.\n\n"
        f"Expected:\n{EXPECTED_DRABF}\n\n"
        f"Actual:\n{drabf_counts}"
    )


print(
    "PASS: DRABF matches CDC."
)


# =================================================================================================
# 20. SAVE ORIGINAL SOURCE COLUMN ORDER
# =================================================================================================

original_day1_columns = (
    day1.columns.tolist()
)


original_day2_columns = (
    day2.columns.tolist()
)


# =================================================================================================
# 21. REMOVE ONLY REDUNDANT DAY 2 SHARED COPIES
# =================================================================================================

#
# Canonical copy:
#
#     DR1TOT_I
#
# The redundant same-name copies from DR2TOT_I are
# removed ONLY AFTER numeric and missing-position validation.
#

day2_for_merge = (

    day2.drop(
        columns=SHARED_COLUMNS
    )

)


print(
    "\nRemoved validated redundant DR2TOT_I copies:"
)


for col in SHARED_COLUMNS:

    print(
        " -",
        col
    )


# =================================================================================================
# 22. VERIFY NO OTHER COLUMN COLLISIONS
# =================================================================================================

remaining_overlap = sorted(

    (
        set(
            day1.columns
        )

        &

        set(
            day2_for_merge.columns
        )
    )

    -

    {
        "SEQN"
    }

)


if remaining_overlap:

    raise ValueError(
        "Unexpected overlapping columns remain before merge:\n"
        + str(
            remaining_overlap
        )
    )


print(
    "PASS: No unexpected column collisions remain."
)


# =================================================================================================
# 23. MERGE DAY 1 + DAY 2
# =================================================================================================

dietary = (

    day1.merge(

        day2_for_merge,

        on="SEQN",

        how="outer",

        validate="one_to_one"

    )

)


#
# Outer merge can generate NaN.
#
# Restore merge-generated NaN to blank text.
#

dietary = (
    dietary.fillna("")
)


print(
    "\nPASS: DR1TOT_I + DR2TOT_I merge completed."
)


# =================================================================================================
# 24. FINAL PARTICIPANT CHECK
# =================================================================================================

if len(dietary) != EXPECTED_ROWS:

    raise ValueError(
        "Merged Dietary row count changed.\n"
        f"Expected: {EXPECTED_ROWS:,}\n"
        f"Actual:   {len(dietary):,}"
    )


if (
    dietary["SEQN"]
    .nunique()
    !=
    EXPECTED_ROWS
):

    raise ValueError(
        "Merged unique SEQN count changed."
    )


if (
    dietary["SEQN"]
    .duplicated()
    .any()
):

    raise ValueError(
        "Merged Dietary file contains duplicate SEQN."
    )


if (
    dietary["SEQN"]
    .eq("")
    .any()
):

    raise ValueError(
        "Merged Dietary file contains blank SEQN."
    )


if (
    dietary["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Merged Dietary SEQN contains '.0'."
    )


print(
    "PASS: Final Dietary file contains "
    "9,544 unique participant SEQN."
)


# =================================================================================================
# 25. DUPLICATE COLUMN NAME CHECK
# =================================================================================================

duplicate_column_names = (

    dietary.columns[
        dietary.columns.duplicated()
    ]

    .tolist()

)


if duplicate_column_names:

    raise ValueError(
        "Duplicate column names found:\n"
        + str(
            duplicate_column_names
        )
    )


print(
    "PASS: No duplicate column names."
)


# =================================================================================================
# 26. ENGINEERED DUPLICATE FIELD CHECK
# =================================================================================================

BAD_DUPLICATE_COLUMNS = [

    "WTDRD1_DAY_2_TOTAL_INTAKE",

    "WTDR2D_DAY_2_TOTAL_INTAKE",

    "DRABF_DAY_2_TOTAL_INTAKE",

    "DRDINT_DAY_2_TOTAL_INTAKE",

    "WTDRD1_X",

    "WTDRD1_Y",

    "WTDR2D_X",

    "WTDR2D_Y",

    "DRABF_X",

    "DRABF_Y",

    "DRDINT_X",

    "DRDINT_Y"

]


existing_bad_columns = [

    col

    for col in BAD_DUPLICATE_COLUMNS

    if col in dietary.columns

]


if existing_bad_columns:

    raise ValueError(
        "Engineered Dietary duplicate fields remain:\n"
        + str(
            existing_bad_columns
        )
    )


print(
    "PASS: No engineered Dietary duplicate fields."
)


# =================================================================================================
# 27. CREATE FINAL LOGICAL COLUMN ORDER
# =================================================================================================

SHARED_ORDER = [

    "SEQN",

    "WTDRD1",

    "WTDR2D",

    "DRABF",

    "DRDINT"

]


DAY1_UNIQUE_COLUMNS = [

    col

    for col in original_day1_columns

    if col not in SHARED_ORDER

]


DAY2_UNIQUE_COLUMNS = [

    col

    for col in original_day2_columns

    if (
        col != "SEQN"
        and
        col not in SHARED_COLUMNS
    )

]


FINAL_COLUMN_ORDER = (

    SHARED_ORDER

    +

    DAY1_UNIQUE_COLUMNS

    +

    DAY2_UNIQUE_COLUMNS

)


# =================================================================================================
# 28. VALIDATE FINAL COLUMN ORDER
# =================================================================================================

if (
    len(FINAL_COLUMN_ORDER)
    !=
    len(set(FINAL_COLUMN_ORDER))
):

    raise ValueError(
        "FINAL_COLUMN_ORDER contains duplicate names."
    )


missing_from_order = (

    set(
        dietary.columns
    )

    -

    set(
        FINAL_COLUMN_ORDER
    )

)


extra_in_order = (

    set(
        FINAL_COLUMN_ORDER
    )

    -

    set(
        dietary.columns
    )

)


if missing_from_order or extra_in_order:

    raise ValueError(
        "Final column-order mismatch.\n\n"
        f"Missing from order:\n"
        f"{sorted(missing_from_order)}\n\n"
        f"Extra in order:\n"
        f"{sorted(extra_in_order)}"
    )


dietary = (
    dietary[
        FINAL_COLUMN_ORDER
    ]
)


print(
    "PASS: Final logical column order created."
)


# =================================================================================================
# 29. DISPLAY FINAL STRUCTURE
# =================================================================================================

print(
    "\n--- FINAL DIETARY STRUCTURE ---"
)


print(
    "Shared fields:",
    len(SHARED_ORDER)
)


print(
    "Day 1 unique fields:",
    len(DAY1_UNIQUE_COLUMNS)
)


print(
    "Day 2 unique fields:",
    len(DAY2_UNIQUE_COLUMNS)
)


print(
    "Total fields:",
    len(dietary.columns)
)


# =================================================================================================
# 30. DAY 1 SOURCE-TO-MERGED EXACT TEXT FIDELITY
# =================================================================================================

print(
    "\n--- DR1TOT_I SOURCE FIDELITY ---"
)


day1_check = (

    day1.merge(

        dietary[
            original_day1_columns
        ],

        on="SEQN",

        how="left",

        suffixes=(
            "_SOURCE",
            "_MERGED"
        ),

        validate="one_to_one"

    )

)


day1_failures = []


for col in original_day1_columns:

    if col == "SEQN":

        continue


    source_values = (
        day1_check[
            f"{col}_SOURCE"
        ]
        .fillna("")
    )


    merged_values = (
        day1_check[
            f"{col}_MERGED"
        ]
        .fillna("")
    )


    mismatch_count = int(

        (
            source_values
            !=
            merged_values
        )

        .sum()

    )


    if mismatch_count > 0:

        day1_failures.append(
            (
                col,
                mismatch_count
            )
        )


if day1_failures:

    raise ValueError(
        "DR1TOT_I values changed during merge:\n"
        + str(
            day1_failures
        )
    )


print(
    "PASS: Every retained DR1TOT_I value "
    "preserved text-for-text."
)


# =================================================================================================
# 31. DAY 2 UNIQUE FIELD SOURCE-TO-MERGED TEXT FIDELITY
# =================================================================================================

print(
    "\n--- DR2TOT_I SOURCE FIDELITY ---"
)


day2_check_columns = (

    ["SEQN"]

    +

    DAY2_UNIQUE_COLUMNS

)


day2_check = (

    day2[
        day2_check_columns
    ]

    .merge(

        dietary[
            day2_check_columns
        ],

        on="SEQN",

        how="left",

        suffixes=(
            "_SOURCE",
            "_MERGED"
        ),

        validate="one_to_one"

    )

)


day2_failures = []


for col in DAY2_UNIQUE_COLUMNS:

    source_values = (
        day2_check[
            f"{col}_SOURCE"
        ]
        .fillna("")
    )


    merged_values = (
        day2_check[
            f"{col}_MERGED"
        ]
        .fillna("")
    )


    mismatch_count = int(

        (
            source_values
            !=
            merged_values
        )

        .sum()

    )


    if mismatch_count > 0:

        day2_failures.append(
            (
                col,
                mismatch_count
            )
        )


if day2_failures:

    raise ValueError(
        "DR2TOT_I values changed during merge:\n"
        + str(
            day2_failures
        )
    )


print(
    "PASS: Every unique DR2TOT_I value "
    "preserved text-for-text."
)


# =================================================================================================
# 32. FINAL SHARED-FIELD CHECK AGAINST DR2TOT_I
# =================================================================================================

print(
    "\n--- FINAL SHARED FIELD CHECK ---"
)


final_shared_check = (

    dietary[
        ["SEQN"] + SHARED_COLUMNS
    ]

    .merge(

        day2[
            ["SEQN"] + SHARED_COLUMNS
        ],

        on="SEQN",

        how="inner",

        suffixes=(
            "_FINAL",
            "_DAY2"
        ),

        validate="one_to_one"

    )

)


for col in SHARED_COLUMNS:

    final_raw = (
        final_shared_check[
            f"{col}_FINAL"
        ]
    )


    day2_raw = (
        final_shared_check[
            f"{col}_DAY2"
        ]
    )


    final_missing = (
        final_raw.eq("")
    )


    day2_missing = (
        day2_raw.eq("")
    )


    missing_mismatch = int(

        (
            final_missing
            !=
            day2_missing
        )

        .sum()

    )


    both_populated = (
        ~final_missing
        &
        ~day2_missing
    )


    final_numeric = pd.to_numeric(

        final_raw.loc[
            both_populated
        ],

        errors="raise"

    )


    day2_numeric = pd.to_numeric(

        day2_raw.loc[
            both_populated
        ],

        errors="raise"

    )


    numeric_mismatch = int(

        (
            ~np.isclose(

                final_numeric.to_numpy(
                    dtype=float
                ),

                day2_numeric.to_numpy(
                    dtype=float
                ),

                rtol=0,

                atol=0

            )
        )

        .sum()

    )


    print(
        f"{col}: "
        f"missing mismatches={missing_mismatch:,}, "
        f"numeric mismatches={numeric_mismatch:,}"
    )


    if missing_mismatch != 0:

        raise ValueError(
            f"{col}: final missing positions differ "
            "from DR2TOT_I."
        )


    if numeric_mismatch != 0:

        raise ValueError(
            f"{col}: final numeric values differ "
            "from DR2TOT_I."
        )


print(
    "PASS: Shared variables remain equivalent "
    "to both Dietary source files."
)


# =================================================================================================
# 33. DIETARY DECIMAL PRESERVATION DIAGNOSTIC
# =================================================================================================

#
# This does NOT change any values.
#
# It simply demonstrates that real decimal tokens remain.
#

REPRESENTATIVE_DECIMAL_FIELDS = [

    "WTDRD1",

    "WTDR2D",

    "DR1TPROT",

    "DR1TCARB",

    "DR1TP204",

    "DR1_320Z",

    "DR2TPROT",

    "DR2TCARB",

    "DR2TP204",

    "DR2_320Z"

]


print(
    "\n--- DIETARY DECIMAL PRESERVATION ---"
)


for col in REPRESENTATIVE_DECIMAL_FIELDS:

    if col not in dietary.columns:

        continue


    populated_count = int(

        dietary[col]
        .ne("")
        .sum()

    )


    decimal_token_count = int(

        (
            dietary[col].ne("")
            &
            dietary[col]
            .str.contains(
                ".",
                regex=False
            )
        )

        .sum()

    )


    print(
        f"{col}: "
        f"populated={populated_count:,}, "
        f"decimal tokens={decimal_token_count:,}"
    )


print(
    "PASS: No Dietary values were rounded "
    "or integerized during the merge."
)


# =================================================================================================
# 34. KNOWN SAS XPORT TINY-ARTIFACT CHECK
# =================================================================================================

TINY_TOKEN = (
    "5.397605346934028e-79"
)


tiny_found = {}


for col in dietary.columns:

    if col == "SEQN":

        continue


    count = int(

        dietary[col]

        .str.lower()

        .eq(
            TINY_TOKEN.lower()
        )

        .sum()

    )


    if count > 0:

        tiny_found[
            col
        ] = count


if tiny_found:

    raise ValueError(
        "Known SAS XPORT tiny-zero artifacts remain "
        "in corrected Dietary input:\n"
        + str(
            tiny_found
        )
    )


print(
    "PASS: No known SAS XPORT tiny-zero artifacts."
)


# =================================================================================================
# 35. SCIENTIFIC-NOTATION DIAGNOSTIC
# =================================================================================================

#
# Diagnostic only.
#
# If a corrected source token legitimately contains scientific
# notation, exact text fidelity remains the controlling rule.
#

scientific_notation_fields = {}


for col in dietary.columns:

    if col == "SEQN":

        continue


    count = int(

        dietary[col]

        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )

        .sum()

    )


    if count > 0:

        scientific_notation_fields[
            col
        ] = count


print(
    "\nScientific-notation source tokens:",
    sum(
        scientific_notation_fields.values()
    )
)


if scientific_notation_fields:

    print(
        "INFO: These source tokens will be preserved AS IS:"
    )


    print(
        scientific_notation_fields
    )


# =================================================================================================
# 36. SAVE FINAL CSV
# =================================================================================================

#
# CRITICAL:
#
# No float_format=
#
# No rounding.
#
# No numeric conversion.
#
# Corrected Dietary text is written AS IS.
#

dietary.to_csv(

    OUTPUT_PATH,

    index=False

)


print(
    "\nSaved final Dietary file:"
)


print(
    OUTPUT_PATH
)


# =================================================================================================
# 37. RE-READ FINAL CSV AS TEXT
# =================================================================================================

final_df = pd.read_csv(

    OUTPUT_PATH,

    dtype=str,

    keep_default_na=False

)


# =================================================================================================
# 38. POST-SAVE STRUCTURE CHECK
# =================================================================================================

if final_df.shape != dietary.shape:

    raise ValueError(
        "Saved CSV shape changed.\n"
        f"Before: {dietary.shape}\n"
        f"After:  {final_df.shape}"
    )


if (
    final_df.columns.tolist()
    !=
    dietary.columns.tolist()
):

    raise ValueError(
        "Saved CSV column structure changed."
    )


print(
    "PASS: Saved CSV structure preserved."
)


# =================================================================================================
# 39. EXACT PRE-SAVE / POST-SAVE TEXT FIDELITY
# =================================================================================================

#
# This is the strongest protection for your requirement:
#
#     Dietary decimal values must appear AS IS.
#

post_save_failures = []


for col in dietary.columns:

    mismatch_count = int(

        (
            dietary[col]
            !=
            final_df[col]
        )

        .sum()

    )


    if mismatch_count > 0:

        post_save_failures.append(
            (
                col,
                mismatch_count
            )
        )


if post_save_failures:

    raise ValueError(
        "CSV text changed during export:\n"
        + str(
            post_save_failures
        )
    )


print(
    "PASS: Exact pre-save/post-save text fidelity."
)


# =================================================================================================
# 40. FINAL SEQN VALIDATION
# =================================================================================================

if (
    final_df["SEQN"]
    .duplicated()
    .any()
):

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    final_df["SEQN"]
    .eq("")
    .any()
):

    raise ValueError(
        "Final CSV contains blank SEQN."
    )


if (
    final_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV contains SEQN '.0'."
    )


if (

    ~final_df["SEQN"]
    .str.fullmatch(r"\d+")

).any():

    raise ValueError(
        "Final CSV contains invalid SEQN formatting."
    )


print(
    "PASS: Final SEQN integrity."
)


# =================================================================================================
# 41. FINAL ENGINEERED-DUPLICATE CHECK
# =================================================================================================

final_bad_columns = [

    col

    for col in BAD_DUPLICATE_COLUMNS

    if col in final_df.columns

]


if final_bad_columns:

    raise ValueError(
        "Engineered duplicate fields remain:\n"
        + str(
            final_bad_columns
        )
    )


print(
    "PASS: No redundant Day 2 shared-variable copies."
)


# =================================================================================================
# 42. FINAL WTDRD1 CDC CHECK
# =================================================================================================

final_wtdrd1 = pd.to_numeric(

    final_df["WTDRD1"]
    .where(
        final_df["WTDRD1"] != "",
        np.nan
    ),

    errors="raise"

)


final_wtdrd1_positive = int(

    (
        final_wtdrd1 > 0
    )

    .sum()

)


final_wtdrd1_zero = int(

    final_wtdrd1

    .eq(0)

    .sum()

)


final_wtdrd1_missing = int(

    final_wtdrd1

    .isna()

    .sum()

)


if final_wtdrd1_positive != 8506:

    raise ValueError(
        "Final WTDRD1 positive-weight count differs."
    )


if final_wtdrd1_zero != 1038:

    raise ValueError(
        "Final WTDRD1 zero count differs."
    )


if final_wtdrd1_missing != 0:

    raise ValueError(
        "Final WTDRD1 missing count differs."
    )


print(
    "PASS: Final WTDRD1:"
)


print(
    "  Positive = 8,506"
)


print(
    "  Zero     = 1,038"
)


print(
    "  Missing  = 0"
)


# =================================================================================================
# 43. FINAL WTDR2D CDC CHECK
# =================================================================================================

final_wtdr2d = pd.to_numeric(

    final_df["WTDR2D"]
    .where(
        final_df["WTDR2D"] != "",
        np.nan
    ),

    errors="raise"

)


final_wtdr2d_positive = int(

    (
        final_wtdr2d > 0
    )

    .sum()

)


final_wtdr2d_zero = int(

    final_wtdr2d

    .eq(0)

    .sum()

)


final_wtdr2d_missing = int(

    final_wtdr2d

    .isna()

    .sum()

)


if final_wtdr2d_positive != 7027:

    raise ValueError(
        "Final WTDR2D positive-weight count differs."
    )


if final_wtdr2d_zero != 1479:

    raise ValueError(
        "Final WTDR2D zero count differs."
    )


if final_wtdr2d_missing != 1038:

    raise ValueError(
        "Final WTDR2D missing count differs."
    )


print(
    "PASS: Final WTDR2D:"
)


print(
    "  Positive = 7,027"
)


print(
    "  Zero     = 1,479"
)


print(
    "  Missing  = 1,038"
)


# =================================================================================================
# 44. FINAL DRDINT CHECK
# =================================================================================================

final_drdint = (

    final_df["DRDINT"]

    .value_counts(
        dropna=False
    )

    .to_dict()

)


if final_drdint != EXPECTED_DRDINT:

    raise ValueError(
        "Final DRDINT distribution differs from CDC.\n\n"
        f"Expected:\n{EXPECTED_DRDINT}\n\n"
        f"Actual:\n{final_drdint}"
    )


print(
    "PASS: Final DRDINT:"
)


print(
    "  Day 1 only       = 1,479"
)


print(
    "  Day 1 + Day 2    = 7,027"
)


print(
    "  Missing          = 1,038"
)


# =================================================================================================
# 45. FINAL DRABF CHECK
# =================================================================================================

final_drabf = (

    final_df["DRABF"]

    .value_counts(
        dropna=False
    )

    .to_dict()

)


if final_drabf != EXPECTED_DRABF:

    raise ValueError(
        "Final DRABF distribution differs from CDC.\n\n"
        f"Expected:\n{EXPECTED_DRABF}\n\n"
        f"Actual:\n{final_drabf}"
    )


print(
    "PASS: Final DRABF:"
)


print(
    "  Yes      = 180"
)


print(
    "  No       = 8,326"
)


print(
    "  Missing  = 1,038"
)


# =================================================================================================
# 46. FINAL TINY-XPORT CHECK
# =================================================================================================

final_tiny_found = {}


for col in final_df.columns:

    if col == "SEQN":

        continue


    count = int(

        final_df[col]

        .str.lower()

        .eq(
            TINY_TOKEN.lower()
        )

        .sum()

    )


    if count > 0:

        final_tiny_found[
            col
        ] = count


if final_tiny_found:

    raise ValueError(
        "Tiny XPORT artifacts remain after CSV save:\n"
        + str(
            final_tiny_found
        )
    )


print(
    "PASS: No known tiny XPORT artifacts after save."
)


# =================================================================================================
# 47. FINAL SUMMARY
# =================================================================================================

print(
    "\n"
    + "=" * 110
)


print(
    "NHANES 2015-2016 DIETARY MERGE COMPLETE"
)


print(
    "=" * 110
)


print(
    "Cycle:"
)


print(
    "  2015-2016"
)


print(
    ""
)


print(
    "Source files:"
)


print(
    "  DR1TOT_I_corrected.csv"
)


print(
    "  DR2TOT_I_corrected.csv"
)


print(
    ""
)


print(
    "Rows:",
    f"{len(final_df):,}"
)


print(
    "Unique SEQN:",
    f"{final_df['SEQN'].nunique():,}"
)


print(
    "Columns:",
    f"{len(final_df.columns):,}"
)


print(
    ""
)


print(
    "CDC Day 1 weight:"
)


print(
    "  Positive WTDRD1 = 8,506"
)


print(
    "  WTDRD1 zero     = 1,038"
)


print(
    "  WTDRD1 missing  = 0"
)


print(
    ""
)


print(
    "CDC Day 2 weight:"
)


print(
    "  Positive WTDR2D = 7,027"
)


print(
    "  WTDR2D zero     = 1,479"
)


print(
    "  WTDR2D missing  = 1,038"
)


print(
    ""
)


print(
    "DRDINT:"
)


print(
    "  Day 1 only       = 1,479"
)


print(
    "  Day 1 + Day 2    = 7,027"
)


print(
    "  Missing          = 1,038"
)


print(
    ""
)


print(
    "DRABF:"
)


print(
    "  Yes              =   180"
)


print(
    "  No               = 8,326"
)


print(
    "  Missing          = 1,038"
)


print(
    ""
)


print(
    "Shared variables retained once:"
)


print(
    "  WTDRD1"
)


print(
    "  WTDR2D"
)


print(
    "  DRABF"
)


print(
    "  DRDINT"
)


print(
    ""
)


print(
    "Duplicate SEQN: 0"
)


print(
    "SEQN '.0': 0"
)


print(
    "Engineered Dietary duplicates: 0"
)


print(
    "Known XPORT tiny artifacts: 0"
)


print(
    "Day 1 retained-value changes: 0"
)


print(
    "Day 2 retained-value changes: 0"
)


print(
    "Dietary decimal reformatting during merge: 0"
)


print(
    "Pre-save/post-save exact text fidelity: PASS"
)


print(
    "=" * 110
)

NHANES 2015-2016 DIETARY DAY 1 + DAY 2 MERGE
DR1TOT_I shape: (9544, 168)
DR2TOT_I shape: (9544, 85)
PASS: SEQN exists in both files.
PASS: Both DR1TOT_I and DR2TOT_I contain 9,544 participant rows.

--- SEQN CHECK: DR1TOT_I ---
Rows: 9,544
Unique SEQN: 9,544
Blank SEQN: 0
Duplicate SEQN: 0
SEQN ending '.0': 0
Invalid SEQN: 0
PASS

--- SEQN CHECK: DR2TOT_I ---
Rows: 9,544
Unique SEQN: 9,544
Blank SEQN: 0
Duplicate SEQN: 0
SEQN ending '.0': 0
Invalid SEQN: 0
PASS

PASS: SEQN integrity validated.

--- PARTICIPANT SET CHECK ---
Only in DR1TOT_I: 0
Only in DR2TOT_I: 0
PASS: Both files contain the same 9,544 participants.

--- SHARED VARIABLES DETECTED ---
['WTDRD1', 'WTDR2D', 'DRABF', 'DRDINT']
PASS: Exactly four expected shared variables found.

--- SHARED VARIABLE FIDELITY CHECK ---

WTDRD1
  Both populated: 9,544
  Missing-position mismatches: 0
  True numeric mismatches: 0
  Text-format differences: 0

WTDR2D
  Both populated: 8,506
  Missing-position mismatches: 0
  True numeric mismat